# Chapter 4 &mdash; $Cond(L)$ and its Negation

**Concept 16 of the Chapter 4 decomposition:** *$Cond(L)$ and its Negation, Written Out*

Negating flips all four quantifiers &mdash; and that flip <b>is</b> the proof recipe.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Cond-And-Its-Negation/Concept-Cond-And-Its-Negation.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$Cond(L)$: **there is** $N$, such that **for any** $w\in L$ with $|w|\ge N$,
**there exists** a split $w=xyz$ ($y\ne\varepsilon$, $|xy|\le N$) such that
**for all** $i\ge0$, $xy^iz\in L$.

$\neg Cond(L)$: **for any** $N$, **for some** $w\in L$ with $|w|\ge N$,
**for all** splits $w=xyz$ ($y\ne\varepsilon$, $|xy|\le N$),
**there exists** $i$ with $xy^iz\notin L$.

Every quantifier flips. The practical reading is a **division of labour**:
the adversary picks $N$ and the split; **you** pick $w$ and $i$.

## 2. Definitions

### The four quantifiers, and who chooses

In [ ]:
print("%-8s %-14s %-10s" % ("in Cond", "in !Cond", "who picks"))
for a,b,who in [("exists N","for all N","adversary"),
                ("for all w","exists w","YOU"),
                ("exists split","for all splits","adversary"),
                ("for all i","exists i","YOU")]:
    print("%-8s %-14s %-10s" % (a,b,who))

### A checker for $\neg Cond$ on a candidate language

In [ ]:
def splits(w, N):
    return [(w[:i], w[i:j], w[j:])
            for i in range(N+1) for j in range(i+1, min(N, len(w))+1) if j > i]

def neg_cond_witness(in_L, make_w, N, imax=4):
    """You pick w = make_w(N); check EVERY split has SOME bad i."""
    w = make_w(N)
    for (x,y,z) in splits(w, N):
        if all(in_L(x + y*i + z) for i in range(imax+1)):
            return None                      # this split survived -> no proof
    return w

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;15.&nbsp;One-Way Implication: Use the Pumping Lemma Only to Disprove Regularity](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-One-Way-Implication/Concept-One-Way-Implication.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;17.&nbsp;Why All Splits of $x,y,z$ Must Be Considered](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Why-All-Splits/Concept-Why-All-Splits.ipynb)&nbsp;&rarr;

---

## 3. Tests

For $L_{01}=\{0^n1^n\}$ choose $w = 0^N1^N$; **every** split fails.

In [ ]:
def in_L01(s):
    k = len(s) - len(s.lstrip('0'))
    return s == '0'*k + '1'*(len(s)-k) and k == len(s)-k

for N in [2,3,4,5]:
    w = neg_cond_witness(in_L01, lambda n: '0'*n + '1'*n, N)
    print("N=%d  witness w = %-14r  every split broken? %s" % (N, w, w is not None))
    assert w is not None

Why $w = 0^N1^N$ works: $|xy|\le N$ forces $y$ to be **all zeros**.

In [ ]:
N = 4; w = '0'*N + '1'*N
ys = {y for (x,y,z) in splits(w, N) if y}
print("all possible y with |xy| <= N :", sorted(ys))
assert all(set(y) <= {'0'} for y in ys)
print("\nEvery y is zeros only -> pumping changes the 0-count but not the 1-count.")

A **badly chosen** $w$ leaves a surviving split and proves nothing.

In [ ]:
bad = neg_cond_witness(in_L01, lambda n: '', 3)
print("w = '' (too short) -> witness:", bad, " <- no proof")
print("\nYou must choose w as a FUNCTION OF N, and long enough.")

## 4. Exercises


1. Write $\neg Cond$ out in predicate logic and check each flip.
2. For $L_{br}=\{\{^i\}^i\}$, which $w$ would you choose?
3. Which two of the four choices are yours? Why does that make the proof feasible?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4/Concept-Cond-And-Its-Negation')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')